# 03 - Construção do corpus e chunking

Esta etapa cria o corpus textual unificado que será usado pelo RAG.

Regras principais:

- usa OCR quando houver um JSON OCR disponível;
- usa extração nativa nos demais artigos;
- mantém `article_id`, ano, páginas e fonte textual em cada chunk;
- cria chunks por tokens com overlap;
- não define uma ordem artificial entre artigos do mesmo ano.

In [1]:
import pandas as pd

from wonca_rag.config import load_experiment_config
from wonca_rag.paths import PATHS
from wonca_rag.rag.chunking import (
    ChunkingSettings,
    build_chunk_corpus,
    save_chunk_corpus,
    summarize_chunk_corpus,
)
from wonca_rag.rag.corpus import (
    build_article_source_table,
    filter_temporal_history,
)

config = load_experiment_config()
MANIFEST_PATH = PATHS.data_root / 'metadata' / 'articles_manifest.parquet'
manifest_df = pd.read_parquet(MANIFEST_PATH)

print('Manifest rows:', len(manifest_df))
print('Unique contents:', manifest_df['sha256'].nunique())

Manifest rows: 233
Unique contents: 215


## 1. Conferir a fonte textual escolhida para cada artigo

In [2]:
source_df = build_article_source_table(
    manifest_df=manifest_df,
    processed_root=PATHS.data_root / 'processed',
)

display(source_df.head(20))
print('\nFontes textuais:')
display(source_df['resolved_text_source'].value_counts(dropna=False))
print('\nErros de fonte:', source_df['source_error'].notna().sum())

,article_id,sha256,filename,relative_path,publication_year,publication_date,resolved_text_source,source_json,source_error
0,ART_8F765D144499,8f765d14449960d97572b150550a93fef5cac5c22f0817...,Abramson2005.pdf,Abramson2005.pdf,2005,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_8F765D1...,None
1,ART_2179244B822A,2179244b822a8bc3cee601b16b87678e023f070a5d61b6...,Allen1989.pdf,Allen1989.pdf,1989,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_2179244...,None
2,ART_404FD0BBD2B8,404fd0bbd2b8fdf55aaf93fb679af45b3115ad4754624d...,Almeida2001.pdf,Almeida2001.pdf,2001,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_404FD0B...,None
3,ART_E58422CEE3D5,e58422cee3d5e0eeb543b2918fe041e7cc1e8d4ecd37b6...,Alonso2008.pdf,Alonso2008.pdf,2008,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E58422C...,None
4,ART_CC8D0D4F8DEE,cc8d0d4f8deed88fc7a21478e72b25555c65c120e35a8a...,Barker1989.pdf,Artigos sem resumo/Barker1989.pdf,1989,NaT,ocr_tesseract,C:\Github\wonca-rag\data\processed\ocr\ART_CC8...,None
5,ART_C51FA4EF984E,c51fa4ef984e20dfc6f6fa26e16ccefdd4b4ba622c8ef6...,Starfield1971b (1).pdf,Artigos sem resumo/Starfield1971b (1).pdf,1971,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_C51FA4E...,None
6,ART_B36F04112348,b36f04112348c77bcb47dfdf9973b840d64740b905da0d...,Starfield1971c.pdf,Artigos sem resumo/Starfield1971c.pdf,1971,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_B36F041...,None
7,ART_0080057343D5,0080057343d5757eaf4f35c839a60a5145cfd9efc93680...,Starfield1976b.pdf,Artigos sem resumo/Starfield1976b.pdf,1976,NaT,ocr_tesseract,C:\Github\wonca-rag\data\processed\ocr\ART_008...,None
8,ART_5235CCA4CEF6,5235cca4cef630f8f9d28b6ffa542834bc011e69f046c8...,Starfield1981d.pdf,Artigos sem resumo/Starfield1981d.pdf,1981,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_5235CCA...,None
9,ART_27019C5BC11E,27019c5bc11e1caee62b4f83e2e19319a89ead2bf0cccb...,Starfield1982c.pdf,Artigos sem resumo/Starfield1982c.pdf,1982,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_27019C5...,None



Fontes textuais:


resolved_text_source
native_pdf_text    204
ocr_tesseract       11
Name: count, dtype: int64


Erros de fonte: 0


## 2. Construir os chunks

In [3]:
settings = ChunkingSettings(
    chunk_size_tokens=config.chunking.chunk_size_tokens,
    chunk_overlap_tokens=config.chunking.chunk_overlap_tokens,
)

print(settings)

ChunkingSettings(chunk_size_tokens=1000, chunk_overlap_tokens=150, encoding_name='cl100k_base')


In [4]:
chunks_df, quality_df = build_chunk_corpus(
    manifest_df=manifest_df,
    processed_root=PATHS.data_root / 'processed',
    settings=settings,
)

print('Chunks:', len(chunks_df))
display(chunks_df.head())

Building chunks:   0%|          | 0/215 [00:00<?, ?it/s]

Chunks: 2215


,chunk_schema_version,article_id,chunk_index,chunk_id,sha256,filename,publication_year,publication_date,text_source,source_json,page_start,page_end,token_start,token_end,n_tokens,text
0,1.0,ART_E3A2395BE56C,0,ART_E3A2395BE56C_C0000_3CB38B626B77,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,Starfield1967.pdf,1967,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E3A2395...,1,2,0,1000,1000,"May, 1967\nThe Journal o/ PEDIATRICS\n777\nFun..."
1,1.0,ART_E3A2395BE56C,1,ART_E3A2395BE56C_C0001_8E5D9B13B8FC,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,Starfield1967.pdf,1967,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E3A2395...,2,3,850,1850,1000,oral\nwater load was arbitrarily chosen as sat...
2,1.0,ART_E3A2395BE56C,2,ART_E3A2395BE56C_C0002_C4CE74F60A3E,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,Starfield1967.pdf,1967,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E3A2395...,3,4,1700,2700,1000,All\nPrimary\nprimary\nAll\nNo.\nI VoI.\nAge\n...
3,1.0,ART_E3A2395BE56C,3,ART_E3A2395BE56C_C0003_3D83073B6AE9,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,Starfield1967.pdf,1967,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E3A2395...,3,5,2550,3550,1000,"Furthermore, a significantly (p<\n0.01) greate..."
4,1.0,ART_E3A2395BE56C,4,ART_E3A2395BE56C_C0004_A3940A0D37DC,e3a2395be56c7e7b3fef9df7fb5524dd7e5236d14ae32f...,Starfield1967.pdf,1967,NaT,native_pdf_text,C:\Github\wonca-rag\data\processed\ART_E3A2395...,5,5,3400,4207,807,those children requiring more than\ntwo\nspeci...


## 3. Resumo de qualidade

In [5]:
summary = summarize_chunk_corpus(chunks_df, quality_df)
display(pd.DataFrame(summary.items(), columns=['metric', 'value']))

print('\nStatus por artigo:')
display(quality_df['status'].value_counts(dropna=False))

print('\nFonte textual por artigo:')
display(quality_df['text_source'].value_counts(dropna=False))

,metric,value
0,n_articles,215.000000
1,n_articles_done,215.000000
2,n_articles_no_chunks,0.000000
3,n_articles_error,0.000000
4,n_articles_ocr,11.000000
5,n_chunks,2215.000000
6,mean_chunks_per_article,10.302326



Status por artigo:


status
DONE    215
Name: count, dtype: int64


Fonte textual por artigo:


text_source
native_pdf_text    204
ocr_tesseract       11
Name: count, dtype: int64

## 4. Inspecionar artigos sem chunks ou com erro

In [6]:
problem_df = quality_df.loc[
    quality_df['status'].isin(['NO_CHUNKS', 'ERROR'])
].copy()

print('Artigos problemáticos:', len(problem_df))
display(problem_df)

Artigos problemáticos: 0


,article_id,filename,publication_year,text_source,source_json,n_pages,total_characters,n_chunks,status,error_message


## 5. Conferir distribuição do tamanho dos chunks

In [7]:
display(chunks_df['n_tokens'].describe())

display(
    chunks_df.groupby('article_id')['chunk_id']
    .count()
    .describe()
)

count    2215.000000
mean      957.832506
std       147.096919
min        60.000000
25%      1000.000000
50%      1000.000000
75%      1000.000000
max      1000.000000
Name: n_tokens, dtype: float64

count    215.000000
mean      10.302326
std        6.858840
min        1.000000
25%        5.000000
50%       10.000000
75%       13.000000
max       51.000000
Name: chunk_id, dtype: float64

## 6. Teste manual da regra temporal

Aqui verificamos que, ao avaliar um artigo de determinado ano, o histórico não contém artigos futuros nem artigos do mesmo ano sem data exata.

In [8]:
years = chunks_df['publication_year'].dropna().astype(int)
test_year = int(years.median())

history_df = filter_temporal_history(
    chunks_df=chunks_df,
    current_year=test_year,
)

print('Ano usado no teste:', test_year)
print('Maior ano recuperado:', history_df['publication_year'].max())
print('Chunks elegíveis:', len(history_df))

assert history_df['publication_year'].dropna().astype(int).lt(test_year).all()
print('Regra temporal: OK')

Ano usado no teste: 2000
Maior ano recuperado: 1999
Chunks elegíveis: 1066
Regra temporal: OK


## 7. Salvar corpus

In [9]:
paths = save_chunk_corpus(
    chunks_df=chunks_df,
    quality_df=quality_df,
    chunks_output_dir=PATHS.artifacts_root / 'chunks',
    logs_output_dir=PATHS.outputs_root / 'logs',
)

for name, path in paths.items():
    print(name, ':', path)

chunks_parquet : C:\Github\wonca-rag\artifacts\chunks\article_chunks.parquet
chunks_csv : C:\Github\wonca-rag\artifacts\chunks\article_chunks.csv
quality_parquet : C:\Github\wonca-rag\outputs\logs\chunking_report.parquet
quality_xlsx : C:\Github\wonca-rag\outputs\logs\chunking_report.xlsx


## Resultado

O arquivo principal desta etapa é `artifacts/chunks/article_chunks.parquet`.

Na próxima etapa, esse corpus será usado para construir o índice semântico e o índice BM25.